In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import pickle
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report

## Load

In [2]:
# Load
file = 'data/raw/holdout.csv'
gateway_tables = pickle.load(open('gateway_tables.pkl', 'rb'))
speed_tables = pickle.load(open('speed_tables.pkl', 'rb'))
in_development = True # enable to immediately judge performance on holdout data

# Make df
test_df = pd.read_csv(file)
if in_development:
    slow_speed_true = (test_df['router_speedtest_tcp_64_download_Mbps_avg'] < 50).astype(int)
    test_df = test_df.drop(columns=['router_speedtest_tcp_64_download_Mbps_avg'])

output_df = test_df.copy() # append the true false column to this at the end


## Transform

In [3]:
numToStr = ['cell_id','home_pop_id','gateway_id','satellite_id', 'ku_beam_target_cell_id', 'link_failure_detected']
for colName in numToStr:
    test_df[colName] = test_df[colName].astype(str)

test_df = test_df.rename(columns={
    'router_speedtest_tcp_64_download_Mbps_avg': 'Speed',
    'slot_timestamp': 'Time_(datetime)',
    'ut_ping_loss_rate': 'Ping_loss',
    'ut_snr': 'SNR',
    'expected_min_snr': 'SNR_base',
    'ut_latency': 'Latency',
    'link_failure_detected': 'Link_failure',
    'ut_sat_elevation_angle_mean': 'Angle',
    'number_of_ku_beam_targets': 'Beam_targets',
    'number_of_users_on_ku_beam': 'Beam_users',
    'number_of_users_on_ku_beam_target': 'Beam_users_target',
    'country_code': 'Country',
    'ku_beam_target_cell_id': 'center_cell_id'
})

test_df['Link_failure'] = test_df['Link_failure'].fillna('Unknown')

test_df['beam_pct_allocation'] = 1 / (test_df['Beam_targets'] * test_df['Beam_users_target'])

test_df = test_df.merge(
    speed_tables['gateway_site_keys'],
    on='gateway_id',
    how='left'
)

for col in ['Country','cell_id','center_cell_id','home_pop_id','satellite_id','gateway_id','gateway_site_id']:
    test_df = test_df.merge(
        speed_tables[col],
        on=col,
        how='left',
        suffixes = ('', f'_{col}')
        )
for col in ['gateway_id','satellite_id','gateway_site_id']:
    test_df = test_df.merge(
        gateway_tables[col],
        on=col,
        how='left',
        suffixes = ('', f'_{col}')
        )

# rename cols missing suffix
test_df = test_df.rename(columns={
    'median_speed': 'median_speed_Country',
    'pct_slow': 'pct_slow_Country',
    'median_users': 'median_users_gateway_id',
    'median_Saturation_avg': 'median_Saturation_avg_gateway_id',
    'avg_beam_count': 'avg_beam_count_gateway_id',
})

# encode link failure 
test_df = pd.get_dummies(test_df, columns=['Link_failure'], drop_first=False, dtype='int64')

features = [
    'Ping_loss',
    'SNR',
    'SNR_base',
    'Latency',
    'Angle',
    'Beam_users',
    'beam_pct_allocation',
    'median_speed_Country',
    'pct_slow_Country',
    'median_speed_cell_id',
    'pct_slow_cell_id',
    'median_speed_center_cell_id',
    'pct_slow_center_cell_id',
    'median_speed_home_pop_id',
    'pct_slow_home_pop_id',
    'median_speed_satellite_id',
    'pct_slow_satellite_id',
    'median_speed_gateway_id',
    'pct_slow_gateway_id',
    'median_speed_gateway_site_id',
    'pct_slow_gateway_site_id',
    'median_users_gateway_id',
    'median_Saturation_avg_gateway_id',
    'avg_beam_count_gateway_id',
    'median_users_satellite_id',
    'median_Saturation_avg_satellite_id',
    'avg_beam_count_satellite_id',
    'median_users_gateway_site_id',
    'median_Saturation_avg_gateway_site_id',
    'avg_beam_count_gateway_site_id',
    'Link_failure_True',
    'Link_failure_Unknown'
]

del_list = []
for col in test_df.columns:
    if col not in features:
        del_list.append(col)

test_df = test_df.drop(columns=del_list)

In [4]:
# confirm columns match model inputs
for i, col in enumerate(test_df.columns):
    if col != features[i]:
        print(f'Out of order: {col}')

## Model

In [5]:
model = XGBClassifier()
model.load_model('xgboost_model.json')
slow_speed_p = model.predict_proba(test_df)[:, 1]
slow_speed = (slow_speed_p > 0.3).astype(int) # tune to adjust for precision vs recall
if in_development:
    print(f1_score(slow_speed_true, slow_speed))
    print(classification_report(slow_speed_true, slow_speed))

0.6060606060606061
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       562
           1       0.45      0.91      0.61        11

    accuracy                           0.98       573
   macro avg       0.73      0.94      0.80       573
weighted avg       0.99      0.98      0.98       573



In [6]:
# Output result table
slow_speed_map = {0:'FALSE', 1: 'TRUE'}
output_df['slow_speed'] = [slow_speed_map[num] for num in slow_speed]
output_df.to_csv('test_output.csv', index=False)
output_df.head()

,slot_timestamp,ut_ping_loss_rate,cell_id,home_pop_id,gateway_id,satellite_id,ut_snr,expected_min_snr,ut_latency,link_failure_detected,ut_sat_elevation_angle_mean,number_of_ku_beam_targets,number_of_users_on_ku_beam,number_of_users_on_ku_beam_target,ku_beam_target_cell_id,local_hour,country_code,test_id,slow_speed
0,2021-12-21T15:26:12Z,0.000000,1050959,12,10158,1288,8.468927,5.640497,62.020656,False,0.446739,1,5,5,1050645,23,AU,1,FALSE
1,2021-12-21T15:06:42Z,0.117603,1051272,13,10053,1238,8.707482,5.358471,81.091325,False,0.543679,1,31,31,1051588,23,AU,2,FALSE
2,2021-12-21T15:05:57Z,0.038403,1050955,12,10158,2485,8.867315,6.762557,94.460829,False,0.446760,1,59,59,1050956,23,AU,3,FALSE
3,2021-12-21T14:08:42Z,0.000000,1050962,12,10053,1762,9.449574,6.403923,71.237235,False,0.486477,1,3,3,1050648,22,AU,4,FALSE
4,2021-12-21T13:58:27Z,0.000000,835617,13,10375,2474,7.331825,5.267703,36.324433,False,0.696587,2,71,49,835617,23,AU,5,FALSE
